# Dimensional Data Modeling Complex Data Type and Cumulation Day 1 Lecture

In this lecture we'll be talking about complex data types like Struct and Arrays.
- Arrays are like lists.
- Structs are like tables.

In the video, Zach talked about how he used these complex data types to shrink ths size of the data. He specifically mentioned using an array of structs at AirBnB.

I personally haven't see an array of structs in my work, but I have seen arrays to reduce row counts significantly. It's a very powerful approach and it can reduce your data's size immensely, which will allow for queries to run that otherwise wouldn't be able to run. Generally, I don't think you want to be doing this unless you had to.

## Dimensions

Dimensions are attributes of an entitry (e.g. identifying variables, date of birth, address)

Dimensions can be:
- Slowly changing (SCD)
- Fixed

## Knowing Your Customer is KEY

When designing a data model, it is crucial you understand who your customer is and what they intend to do with your data. If you're customer is an analyst / data scientist, the data model should be flat and easy to query. If you're customer is a data engineer, complex / nested types are ok.

## OLTP vs master data vs OLAP

- OLTP (online transaction processing)
    - optimizes low latency, low volume queries
    - generally what software engs use for apps
- OLAP (online analytical processing)
    - optimizes for large volume (less tables)
- Master data
    - middle ground between oltp and olap

A good way to know the difference between OLTP and OLAP is asking how much data would be pulled for a query. If it's 1 user, then it's OLTP.

Here's the continuum:

production database snapshots --> master data --> olap cubes --> metrics

Zach talked about how at AirBnB, he worked on master data for pricing and availability. He took 40 production database snapshot tables and combined it into 1 for master data. This is a huge value unlock since otherwise, all downstream queries would have to do a lot of joining.

## Cumulative Table Design

Core component of cumulative table design:
- 2 dataframes (yesterday and today)
- full outer join
- coalesce values

Usage - growth analytics at facebook (dim_all_users)

Strengths:
- no need for group by (can easily see someone was active 10 days ago using the facebook example)

Drawbacks:
- can only be backfilled sequentially
- handling pii can be a mess

## Run length encoding

During the lecture, Zach talks about how run length encoding (think parquet) compresses the data significantly. And so you can achieve massive compression as a data engineer if you sort properly. The issue however is that downstream users will run joins on the data, which completely gets rid of the sorting. Therefore, this is something to definitely watch out for. In his example, I believe he opted for a more complex data type to reduce rows instead of trying to achieve compression through sorting.

## Lab

In the lab, we used the player_seasons table to create a cumulative table. It was pretty wild how he created a array of structs. It was a really powerful pattern for sure.

# Dimensional Data Modeling: Building Slowly Changing Dimensions Day 2 Lecture

Slowly changing dimensions are attributes that can change. He talks about favorite food as an example, but I think address is a good real world example.

Modeling scd attributes is important to achieve idempotency, which according to him is pipelines produces the same results regardless of when it's ran. Pipelines should produce the same results regardless of the day you run it, how many times you run it, and regardless of the hour you run it.

More info about idempotent:

He talked about how you should never use INSERT INTO. You should always use MERGE or INSERT OVERWRITE. INSERT INTO exposes you to risk.

After talking about idempotentcy, he talks about SCD how it's important to model them properly. It's interesting because in my own work working at Kroger, our products information is actually a slowing changing dimension. Imagine if you have brand A and manufacturer A that gets purchased by manufacturer B, that item then needs to change to manufacturer B. In our case, our entire analysis structure is based on latest snapshot. Technically, all of our data isn't idempotent.

How to model scd?
- latest snapshots (he said never do this haha)
- daily partitioned snapshots (everyday there's a value for a dimension so grain is at daily level)
- SCD types 0,1,2,3

SCD types:
- Type 0: value doesn't change (date of birth)
- Type 1: you only care about latest value (don't use this type)
- Type 2: (gold standard) dimensions have a start date and end date. Rows with null end date are the current value
- Type 3: you only care about original value and current value (this doesn't seem very useful)

Type 0 and 2 are idempotent. Type 1 and 3 aren't.

## Lab

In this lab, we use the players table to model SCD values via type 2.

# Dimensional Data Modeling: Graph Data Modeling Day 3 Lecture

Why whould you use enums?
- Built in data quality
- Built in static fields
- Built in documentation

Should only be used <50 distinct values

He goes pretty deep into enums and it can be utilized for managing pipelines where there's a lot of different data sources that's getting mapped into the same schema.

He stated best way to merge data sources together is to use a flexible schema via map.

Graph modeling is relationship focused, not entity focused.

Graphs modeling always looks like:
- identifier
- type
- properties map <string, string>

Example:
- Subject identifier: string
- Subject type: vertex type
- object identifier: string
- objecty type: vertex type
- edge type: edge type
- properties: map

Vertices are objects like player, team, etc. Plays on is an edge.

## Lab

This lab we'll be building a graph database using players data.

# Summary

We covered a lot in these 3 lectures. But at a high level, zach walked through how to create data models for:
- Cumulative table designs - he showed us how to use this data model to create very efficient data models
- Modeling slowly changing dimensions - he showed us this to ensure our data pipelines are idempotent. Accounting for SCD is very important in his view, which I agree. However, in my work, most of the dimensional tables i've seen have been as-is.
- Modeling a graph database - this was pretty wild as it was my first time being exposed to these types of data models. It's good for modeling relationships. I think he's said he's only done this 3 times in his career so I probably won't focus on understanding all the details here, but he said it may come up in interviews.